# EWSNet Retinal Vessel Segmentation on Google Colab

This notebook allows you to run the EWSNet model for retinal vessel segmentation using a dataset stored in your Google Drive.

## 1. Mount Google Drive

Mount your Google Drive to access the dataset and save results.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install and Configure Dependencies

Clone the EWSNet repository and install the required packages.

In [ ]:
!git clone https://github.com/xuecheng990531/EWSNet.git
%cd EWSNet
!pip install -r requirements.txt

## 3. Choose Dataset

原作者训练流程默认使用 `chasedb1` 数据集。模型加载代码也支持 `chasedb1`、`stare`、`hrf`、`uwf` 四种数据集。请根据你 Google Drive 中的目录结构选择一个数据集名称。

In [ ]:
import os

# Supported dataset names: chasedb1, stare, hrf, uwf
# 原作者默认训练数据集为 chasedb1

dataset_name = 'chasedb1'  # 修改为你要训练的实际数据集名称

dataset_root = f'/content/drive/MyDrive/{dataset_name}'
train_data_dir = os.path.join(dataset_root, 'train')
test_data_dir = os.path.join(dataset_root, 'test')

print('Dataset name:', dataset_name)
print('Train data dir:', train_data_dir)
print('Test data dir:', test_data_dir)

assert os.path.exists(train_data_dir), f'Train path not found: {train_data_dir}'
assert os.path.exists(test_data_dir), f'Test path not found: {test_data_dir}'

## 3. Load EWSNet Model

Import the model classes and load pre-trained weights if available.

In [ ]:
import torch
from models.thick_net import ThickNet
from models.resunet import ResUnet
from models.refine import denosing_module

# Load models (adjust paths as needed)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model1 = ThickNet().to(device)
model2 = ResUnet().to(device)
refine = denosing_module().to(device)

# Load pre-trained weights if available
# model1.load_state_dict(torch.load('/content/drive/MyDrive/ewsnet_model1.pth'))
# model2.load_state_dict(torch.load('/content/drive/MyDrive/ewsnet_model2.pth'))
# refine.load_state_dict(torch.load('/content/drive/MyDrive/ewsnet_refine.pth'))

model1.eval()
model2.eval()
refine.eval()

## 4. Load Dataset from Google Drive

Set the paths to your dataset in Google Drive and prepare the data loader.

In [ ]:
from torch.utils.data import DataLoader
from tools.dl import fundus_data

# Create data loaders using the selected dataset
train_dataset = fundus_data(train_data_dir, mode='train', name=dataset_name, isnoise=False)
test_dataset = fundus_data(test_data_dir, mode='test', name=dataset_name, isnoise=True)

train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=0)

## 5. Run Model Training

Execute the training process. 原作者训练调用方式会指定 `--test_data_name` 来区分数据集，同时 `train_dataset` 和 `test_dataset` 对应 `isnoise=False` / `isnoise=True`。

In [ ]:
# Run training (adjust paths and parameters)
!python main.py --train_data_dir "$train_data_dir" --test_data_dir "$test_data_dir" \
    --test_data_name "$dataset_name" --gpu_index 0 --epochs 10 --batch_size 2 --num_works 0

## 6. Save and View Results

Save trained models and outputs to Google Drive, and visualize some results.

In [ ]:
# Copy results to Drive
!cp -r ckpts /content/drive/MyDrive/
!cp -r visual_output /content/drive/MyDrive/

# Display some images (if matplotlib is available)
import matplotlib.pyplot as plt
# Assuming some output images are saved
# plt.imshow(plt.imread('visual_output/some_image.png'))
# plt.show()